# 08 — RQ1, chiến lược C: LLM few-shot (tầng 1 + tầng 2)

Chiến lược C không huấn luyện mô hình: em gửi từng review cho `gpt-4o-mini` (snapshot cố định `gpt-4o-mini-2024-07-18`, temperature 0)
kèm một prompt gồm định nghĩa nhãn, cây quyết định, các quy tắc ca khó trong guideline v1.2 và một bộ ví dụ cố định. Notebook này chạy
trên máy (chỉ gọi API, không cần GPU) và chỉ dùng Train (để lấy ví dụ) và Validation (để đánh giá). Hold-out và focused set để dành cho
Giai đoạn 10.

Theo protocol (Q4), ví dụ few-shot chỉ lấy từ Train: tầng 1 có 4 ví dụ ACTIONABLE và 4 ví dụ NOT_ACTIONABLE, tầng 2 có 3 ví dụ cho mỗi
lớp BUG, FEATURE, UX. Em rút ngẫu nhiên một lần với seed 42 (bỏ review ambiguous và review quá ngắn hoặc quá dài), thay vì tự chọn tay,
để tránh chọn những ví dụ "đẹp". Prompt, danh sách ví dụ và `prompt_hash` được khóa trong `04_prompts/` trước khi đánh giá. Repository
không lưu text review, nên trong `04_prompts/` em chỉ ghi review_id, nhãn và mã băm của text; khi chạy, text được đọc lại từ Train và
kiểm tra khớp mã băm.

Chạy theo kiểu cascade: tầng 1 phân mọi review Validation thành ACTIONABLE hoặc NOT_ACTIONABLE; tầng 2 chỉ được gọi cho review là
actionable theo gold hoặc theo dự đoán của tầng 1. Tầng 2 được chấm trên phần gold-actionable (giống ViSoBERT và Bi-LSTM ở Giai đoạn 7),
còn dự đoán trên các review tầng 1 gọi nhầm là actionable được lưu lại cho việc ghép hai tầng ở Giai đoạn 10. LLM không có ngưỡng xác
suất nên em báo thẳng nhãn nó trả về. LLM cũng không được huấn luyện nên không có seed như ViSoBERT hay Bi-LSTM; dù temperature bằng
0, kết quả vẫn có thể không hoàn toàn giống nhau giữa các lần gọi, và em báo kết quả của một lần chạy.

In [1]:
import os, sys, json, datetime
import numpy as np, pandas as pd
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, '02_src'))
from src.utils.seeds import set_seed
set_seed(42)
DATA = os.path.join(ROOT, '01_data', 'processed')
train_df = pd.read_csv(os.path.join(DATA, 'shopee_train.csv'), encoding='utf-8-sig')
val_df = pd.read_csv(os.path.join(DATA, 'shopee_validation.csv'), encoding='utf-8-sig')
assert not set(train_df['review_id']) & set(val_df['review_id'])
OUT = os.path.join(ROOT, '05_experiments', 'rq1_classification')
print('Train', len(train_df), '| Validation', len(val_df), '| kết quả ghi vào', os.path.relpath(OUT, ROOT))

Train 1200 | Validation 350 | kết quả ghi vào 05_experiments\rq1_classification


## Kiểm tra API

Trước khi chạy, em gọi API một lần để chắc snapshot vẫn dùng được và model thực sự trả lời đúng là `gpt-4o-mini-2024-07-18`. API key và
giá token đọc từ `.env` (không ghi trong notebook), key được che trong mọi thông báo lỗi.

In [2]:
from src.utils.llm import check_api, get_client, prices
r = check_api()
assert r['ok'], f"API lỗi: {r.get('error')} — {r.get('hint', '')}"
print('Model trả về:', r['response_model'], '| khớp snapshot:', r['snapshot_match'], '| temperature', r['temperature'])
assert r['snapshot_match'], 'Model trả về khác snapshot đã khóa'
PRICES = prices()
print('Giá / 1M token:', PRICES)
client, SETTINGS = get_client()

Model trả về: gpt-4o-mini-2024-07-18 | khớp snapshot: True | temperature 0.0
Giá / 1M token: {'input': 0.15, 'output': 0.6, 'source': 'env'}


## Prompt và bộ ví dụ (khóa)

Cell dưới rút bộ ví dụ từ Train, ghi danh sách vào `04_prompts/classification_fewshot/`, dựng prompt hoàn chỉnh và đăng ký `prompt_hash`
vào `04_prompts/prompts_registry.md`. Nếu prompt đã được khóa từ trước thì cell chỉ kiểm tra hash có khớp không; prompt khác bản đã khóa
sẽ báo lỗi, muốn đổi thì phải tạo version mới. Em in system prompt và nhãn của từng ví dụ, không in text review.

In [3]:
from src.classification.strategy_llm_fewshot.prompts import select_exemplars, write_manifest, load_prompt, register
VERSION, TODAY = 'v1', datetime.date.today().strftime('%d/%m/%Y')
PROMPTS = {}
for tier in (1, 2):
    write_manifest(select_exemplars(train_df, tier, seed=42), tier, VERSION)
    PROMPTS[tier] = load_prompt(tier, VERSION, train_df)
    h, status = register(PROMPTS[tier], SETTINGS, TODAY)
    ex = PROMPTS[tier].exemplars
    print(f'=== Tầng {tier} | prompt_hash {h[:16]}… ({status}) | {len(ex)} ví dụ:',
          ', '.join(f"{e['label']}({e['gold_label']})" for e in ex))
    print(PROMPTS[tier].system, '\n')

=== Tầng 1 | prompt_hash 9730222c472ff4a7… (khớp bản đã khóa) | 8 ví dụ: ACTIONABLE(BUG), NOT_ACTIONABLE(OTHER), ACTIONABLE(FEATURE), NOT_ACTIONABLE(OTHER), ACTIONABLE(UX), NOT_ACTIONABLE(OTHER), ACTIONABLE(BUG), NOT_ACTIONABLE(OTHER)
Bạn là chuyên gia phân tích yêu cầu phần mềm. Bạn đọc review tiếng Việt của người dùng ứng dụng thương mại điện tử Shopee trên Google Play và quyết định review đó có ACTIONABLE với đội phát triển app hay không.

ACTIONABLE: review chỉ ra một việc CỤ THỂ mà đội phát triển app xử lý được, thuộc một trong ba loại:
- BUG: chức năng/hành vi được kỳ vọng nhưng bị lỗi: crash, đứng, lag, không hoàn thành thao tác, sai kết quả, không tải, không đăng nhập được, hiển thị sai.
- FEATURE: đề xuất chức năng mới, hoặc thay đổi/cải tiến cụ thể đối với chức năng đã có.
- UX: chức năng về cơ bản vẫn hoạt động nhưng cách trình bày/tương tác gây khó hiểu, khó thao tác hoặc bất tiện cụ thể.
NOT_ACTIONABLE: khen/chê chung chung; không chỉ ra việc cụ thể; vấn đề giao hàng, ship

## Chạy thử 20 review

Trước khi chạy toàn bộ Validation, em chạy thử 20 review (10 actionable và 10 OTHER theo gold, rút với seed 42) để kiểm tra prompt, cách
parse kết quả và chi phí thật, rồi ước lượng chi phí cho 350 review. Kết quả chạy thử không dùng để đánh giá.

In [4]:
from src.classification.strategy_llm_fewshot.classify import classify, cost_summary, results_from_log
is_act = val_df['gold_label'] != 'OTHER'
pilot = pd.concat([val_df[is_act].sample(10, random_state=42), val_df[~is_act].sample(10, random_state=42)])
PILOT_LOG = os.path.join(OUT, 'llm_pilot_calls.jsonl')
if os.path.exists(PILOT_LOG):              # đã chạy thử: dựng lại từ log, không gọi API lần nữa
    print('Đã chạy thử trước đó — đọc lại từ', os.path.relpath(PILOT_LOG, ROOT), '(cột giây không có trong log)')
    p1, p2 = results_from_log(PILOT_LOG, 1), results_from_log(PILOT_LOG, 2)
else:
    p1 = classify(pilot, PROMPTS[1], client, SETTINGS, PILOT_LOG)
    t2_ids = set(pilot.loc[pilot['gold_label'] != 'OTHER', 'review_id']) | set(p1.loc[p1['pred_label'] == 'ACTIONABLE', 'review_id'])
    p2 = classify(pilot[pilot['review_id'].isin(t2_ids)], PROMPTS[2], client, SETTINGS, PILOT_LOG)
show = pilot[['review_id', 'gold_label']].merge(p1[['review_id', 'pred_label']].rename(columns={'pred_label': 'tầng 1'}), on='review_id') \
    .merge(p2[['review_id', 'pred_label']].rename(columns={'pred_label': 'tầng 2'}), on='review_id', how='left').fillna({'tầng 2': '—'})
print(show.assign(review_id=lambda d: d['review_id'].str[:8]).to_string(index=False))
pc = pd.DataFrame([cost_summary(p1, PRICES, 'tầng 1'), cost_summary(p2, PRICES, 'tầng 2')])
print(pc.round(5).to_string(index=False))
act_rate = (p1['pred_label'] == 'ACTIONABLE').mean()
est = len(val_df) * (pc.loc[0, 'usd_/1000_review'] + max(act_rate, 0.25) * pc.loc[1, 'usd_/1000_review']) / 1000
print(f'Ước lượng chi phí cho {len(val_df)} review Validation (tầng 2 cho khoảng {max(act_rate, 0.25):.0%} review): ~${est:.3f}')

Đã chạy thử trước đó — đọc lại từ 05_experiments\rq1_classification\llm_pilot_calls.jsonl (cột giây không có trong log)
review_id gold_label         tầng 1 tầng 2
 7eb7cc1e        BUG     ACTIONABLE    BUG
 01682bdb         UX     ACTIONABLE     UX
 e8e1bc81         UX     ACTIONABLE     UX
 0bbc94d6         UX     ACTIONABLE     UX
 ff9ece08         UX     ACTIONABLE     UX
 6f55e849        BUG     ACTIONABLE    BUG
 41abaa02         UX     ACTIONABLE     UX
 986e1798         UX     ACTIONABLE     UX
 e661c118        BUG     ACTIONABLE    BUG
 44364696         UX     ACTIONABLE     UX
 1d7faffe      OTHER NOT_ACTIONABLE      —
 6becb565      OTHER NOT_ACTIONABLE      —
 bb0a1292      OTHER NOT_ACTIONABLE      —
 7922ac51      OTHER NOT_ACTIONABLE      —
 e66142e1      OTHER NOT_ACTIONABLE      —
 da5796a1      OTHER NOT_ACTIONABLE      —
 504a3ed9      OTHER NOT_ACTIONABLE      —
 bfa18d81      OTHER NOT_ACTIONABLE      —
 eb565b64      OTHER NOT_ACTIONABLE      —
 7cb93df3      OTHER

## Chạy toàn bộ Validation

Cell này chỉ chạy khi `CONFIRM_FULL_RUN = True`, sau khi em đã xem kết quả chạy thử và được xác nhận. Để tránh gọi API hai lần, cell
dừng lại nếu file dự đoán đã tồn tại.

In [5]:
CONFIRM_FULL_RUN = True       # đã xác nhận sau khi xem chạy thử (prompt v1)
P1_CSV, P2_CSV = os.path.join(OUT, 'tier1_llm_val_predictions.csv'), os.path.join(OUT, 'tier2_llm_val_predictions.csv')
FULL_LOG = os.path.join(OUT, 'llm_calls.jsonl')
if not CONFIRM_FULL_RUN:
    print('Chưa chạy toàn bộ Validation (CONFIRM_FULL_RUN = False).')
elif os.path.exists(P1_CSV):
    print('Đã có', os.path.relpath(P1_CSV, ROOT), '— không gọi API lại.')
else:
    r1 = classify(val_df, PROMPTS[1], client, SETTINGS, FULL_LOG)
    r1 = val_df[['review_id', 'gold_label', 'ambiguous_flag']].merge(r1, on='review_id')
    r1['gold_y'] = (r1['gold_label'] != 'OTHER').astype(int)
    r1['prompt_hash'] = PROMPTS[1].hash(SETTINGS)
    r1.to_csv(P1_CSV, index=False)
    ids2 = set(r1.loc[r1['gold_y'] == 1, 'review_id']) | set(r1.loc[r1['pred_label'] == 'ACTIONABLE', 'review_id'])
    r2 = classify(val_df[val_df['review_id'].isin(ids2)], PROMPTS[2], client, SETTINGS, FULL_LOG)
    r2 = val_df[['review_id', 'gold_label', 'ambiguous_flag']].merge(r2, on='review_id')
    r2['gold_actionable'] = r2['gold_label'] != 'OTHER'
    r2['tier1_pred_actionable'] = r2['review_id'].isin(r1.loc[r1['pred_label'] == 'ACTIONABLE', 'review_id'])
    r2['prompt_hash'] = PROMPTS[2].hash(SETTINGS)
    r2.to_csv(P2_CSV, index=False)
    print('Đã lưu dự đoán:', os.path.relpath(P1_CSV, ROOT), os.path.relpath(P2_CSV, ROOT))

Đã có 05_experiments\rq1_classification\tier1_llm_val_predictions.csv — không gọi API lại.


## Kết quả trên Validation

Tầng 1 được chấm trên toàn bộ 350 review Validation, tầng 2 trên 71 review gold-actionable (giữ cả review ambiguous). Em đặt kết quả
của LLM cạnh ViSoBERT và Bi-LSTM từ Giai đoạn 6 và 7 chỉ để tham khảo. Hai mô hình kia là trung bình ba seed, ở cấu hình và ngưỡng đã
chọn trên chính Validation, nên con số của chúng lạc quan hơn; LLM không được tune trên Validation. So sánh chính thức để ở Giai đoạn 10
trên Hold-out. Chi phí của LLM là chi phí thật theo số token API trả về; LLM không cần GPU và không cần huấn luyện.

In [6]:
from src.classification.learning_curve import tier1_metrics
from src.classification.tier2_bfu.evaluate import metrics as tier2_metrics
if not os.path.exists(P1_CSV):
    print('Chưa có dự đoán đầy đủ trên Validation.')
else:
    r1, r2 = pd.read_csv(P1_CSV), pd.read_csv(P2_CSV)
    m1 = tier1_metrics(r1['gold_y'], (r1['pred_label'] == 'ACTIONABLE').astype(int))
    g2 = r2[r2['gold_actionable']]
    m2 = tier2_metrics(g2['gold_label'].map({'BUG': 0, 'FEATURE': 1, 'UX': 2}), g2['pred'])
    pct = lambda v: f'{v * 100:.1f}'
    t1 = pd.read_csv(os.path.join(OUT, 'tier1_validation_results.csv'))
    t1 = pd.concat([t1[['strategy', 'f1_actionable', 'recall_actionable', 'precision_actionable', 'macro_f1']],
                    pd.DataFrame([{'strategy': 'LLM few-shot (1 lần chạy)', **{k: pct(v) for k, v in m1.items()}}])])
    t2 = pd.read_csv(os.path.join(OUT, 'tier2_validation_results.csv'))
    t2 = pd.concat([t2[['strategy', 'macro_f1', 'f1_BUG', 'f1_FEATURE', 'f1_UX']],
                    pd.DataFrame([{'strategy': 'LLM few-shot (1 lần chạy)', **{k: pct(m2[k]) for k in ('macro_f1', 'f1_BUG', 'f1_FEATURE', 'f1_UX')}}])])
    print(f'Tầng 1 trên Validation (n = {len(r1)}; %):'); print(t1.to_string(index=False))
    print(f'\nTầng 2 trên Validation-actionable (n = {len(g2)}; %):'); print(t2.to_string(index=False))
    print('\nP/R LLM tầng 2:', {k: round(v, 3) for k, v in m2.items() if k.startswith(('precision', 'recall'))})
    cost = pd.DataFrame([cost_summary(r1, PRICES, 'tầng 1 (mọi review)'), cost_summary(r2, PRICES, 'tầng 2 (actionable gold ∪ dự đoán)')])
    cost.to_csv(os.path.join(OUT, 'llm_costs.csv'), index=False)
    print('\nChi phí thật của LLM (giá từ .env; token cached tính đủ giá — cận trên):')
    print(cost.round(4).to_string(index=False))
    print(f"Tổng: ${cost['chi_phí_usd'].sum():.4f} | GPU: 0 | huấn luyện: 0 | model trả về:",
          sorted({json.loads(l)['model'] for l in open(FULL_LOG, encoding='utf-8')}))
    pd.DataFrame([{'tầng': 'tier1', **m1}, {'tầng': 'tier2', **m2}]).to_csv(os.path.join(OUT, 'llm_validation_metrics.csv'), index=False)

Tầng 1 trên Validation (n = 350; %):
                 strategy f1_actionable recall_actionable precision_actionable   macro_f1
                 ViSoBERT    76.5 ± 0.6        82.6 ± 4.3           71.6 ± 4.1 85.0 ± 0.6
       Bi-LSTM + fastText    68.5 ± 4.2        87.8 ± 6.4           56.9 ± 8.3 78.6 ± 3.9
LLM few-shot (1 lần chạy)          82.7              87.3                 78.5       89.0

Tầng 2 trên Validation-actionable (n = 71; %):
                 strategy   macro_f1     f1_BUG f1_FEATURE      f1_UX
                 ViSoBERT 75.6 ± 2.2 74.2 ± 2.6 73.1 ± 2.7 79.5 ± 2.7
       Bi-LSTM + fastText 72.0 ± 3.2 72.3 ± 3.2 63.5 ± 5.7 80.3 ± 1.3
LLM few-shot (1 lần chạy)       79.9       81.5       73.2       85.1

P/R LLM tầng 2: {'precision_BUG': 0.815, 'recall_BUG': 0.815, 'precision_FEATURE': 0.789, 'recall_FEATURE': 0.682, 'precision_UX': 0.8, 'recall_UX': 0.909}

Chi phí thật của LLM (giá từ .env; token cached tính đủ giá — cận trên):
                              bước  n_review

## Nhận xét

Prompt v1 chạy ổn định trên toàn bộ Validation: 438 lần gọi API (350 ở tầng 1 và 88 ở tầng 2), không lần nào phải thử lại vì trả lời
sai định dạng, không có fallback, không bị từ chối, và mọi lần gọi đều trả về đúng snapshot `gpt-4o-mini-2024-07-18`. Tuy vậy, trong
cùng một lần chạy, phía OpenAI trả về năm giá trị `system_fingerprint` khác nhau, tức là cấu hình phía máy chủ thay đổi giữa các lần
gọi. Đây là thêm một lý do để em coi kết quả của LLM là kết quả của một lần chạy, không lặp lại được hoàn toàn như mô hình có seed.

Ở tầng 1, LLM đạt F1 ACTIONABLE 82,7, recall 87,3, precision 78,5 và Macro-F1 89,0. Trên 71 review actionable, LLM bỏ sót 9 review
(6 BUG, 3 FEATURE) và gọi nhầm 17 review OTHER là actionable. Các con số này cao hơn ViSoBERT (76,5 / 82,6 / 71,6 / 85,0) và Bi-LSTM,
nhưng em chưa so sánh trực tiếp: hai mô hình kia là trung bình ba seed ở cấu hình và ngưỡng chọn trên chính Validation, còn LLM là một
lần chạy, không có ngưỡng và prompt không được chỉnh theo Validation. Dù không có ngưỡng để chỉnh, recall của LLM vẫn vượt sàn 0,80
của tầng 1.

Ở tầng 2, LLM đạt Macro-F1 79,9 (BUG 81,5, FEATURE 73,2, UX 85,1). F1 FEATURE gần như bằng ViSoBERT (73,1), recall FEATURE cũng bằng
(15 trên 22 review); phần LLM cao hơn đến từ BUG và UX. Trên 71 review, mỗi review đổi nhãn làm Macro-F1 thay đổi khoảng 1–2 điểm, nên
chênh lệch vài điểm ở đây chỉ tương ứng với vài review.

Em soi riêng cặp BUG/FEATURE, ranh giới khó nhất ở Giai đoạn 7. LLM nhầm 7 review giữa hai lớp này (4 FEATURE thành BUG, 3 BUG thành
FEATURE), so với trung bình 8,0 review mỗi lần chạy của ViSoBERT và 9,7 của Bi-LSTM. Lỗi FEATURE thành BUG ít hơn (4 so với 5,0 và
7,7), nhưng BUG/FEATURE vẫn là cặp LLM nhầm nhiều nhất (BUG/UX 3 review, FEATURE/UX 4 review). Đọc các ca này, em thấy LLM bị kéo theo
từ ngữ bề mặt dù quy tắc R6 có trong prompt: một review có chữ "lỗi" bị gọi là BUG dù thực chất là đề nghị bỏ một chức năng; các review
than "không có chỗ" hoặc "không cho" làm một thao tác mà app không hỗ trợ (theo R6 là FEATURE) bị gọi là BUG; ngược lại, review than một
chức năng có sẵn không chạy đúng (theo R6 là BUG) lại bị gọi là FEATURE, thường vì có cụm "không cho". Đây cũng là những ca khó với hai
mô hình kia: trên 7 review này, ViSoBERT chỉ đúng 11 trên 21 lượt (ba seed) và Bi-LSTM 13 trên 21.

Một chi tiết cần nhớ cho bước ghép hai tầng ở Giai đoạn 10: trong 17 review OTHER mà tầng 1 gọi nhầm là actionable, tầng 2 gán 12 review
là BUG, 4 là FEATURE và 1 là UX. Nếu chạy cascade, nhiễu từ tầng 1 sẽ dồn chủ yếu vào lớp BUG.

Về chi phí, toàn bộ lần chạy tốn $0,0749 (tầng 1 $0,0607 cho 350 review, tầng 2 $0,0142 cho 88 review), tức khoảng $0,17 cho mỗi 1.000
review ở mỗi tầng, với khoảng 488 nghìn token vào và 2,8 nghìn token ra. Con số này tính mọi token vào theo giá đầy đủ nên là cận trên:
89% token vào của tầng 1 được cache và được tính giá thấp hơn. LLM không cần GPU và không cần huấn luyện. Đổi lại, gọi tuần tự mất
khoảng 0,9 giây mỗi review (khoảng 934 giây cho 1.000 review), trong khi ViSoBERT dự đoán 1.000 review trong 1,5 giây trên GPU sau khi đã
huấn luyện (khoảng 25 giây mỗi lần ở tầng 1). Với mỗi 10.000 review, tầng 1 của LLM tốn khoảng $1,7 và khoảng 2,6 giờ nếu gọi tuần tự;
gọi song song sẽ rút ngắn thời gian. Ngoài ra, LLM chỉ cần 17 review gán nhãn làm ví dụ, còn ViSoBERT và Bi-LSTM cần 1.200 review Train;
cả ba chiến lược đều cần Validation để đánh giá.

Kết quả của LLM có một hạn chế riêng từ bộ ví dụ. Bộ ví dụ few-shot được rút ngẫu nhiên từ Train với seed 42 và khóa trước khi đánh
giá. Vì rút ngẫu nhiên nên có một ví dụ FEATURE khá mơ hồ, đọc như một lời khen, và các ví dụ UX đều nói về quảng cáo nên khá hẹp. Đây là
đánh đổi em chấp nhận để giữ khách quan: em không chọn tay hay đổi ví dụ sau khi xem kết quả chạy thử. LLM few-shot nhạy với chất lượng
ví dụ, nên kết quả ở đây cần hiểu là kết quả "với một bộ ví dụ ngẫu nhiên hợp lý", không phải "với bộ ví dụ tối ưu".

Em chưa kết luận chiến lược nào tốt hơn. So sánh chính thức của RQ1 là trên Hold-out ở Giai đoạn 10, tính cả chi phí. Dự đoán của LLM
trên Validation đã được lưu trong `05_experiments/rq1_classification/` (`tier1_llm_val_predictions.csv`, `tier2_llm_val_predictions.csv`),
cùng log từng lần gọi API (`llm_calls.jsonl`, không chứa text review) và bảng chi phí (`llm_costs.csv`).